# Step 3 — Import Data ke Neo4j
Membuat node dan relationship di database `flimkg` menggunakan Python neo4j driver.

```
Node     : Movie, Actor, Director, Genre, Keyword
Relations: actedBy, directedBy, hasGenre, hasKeyword
```

## Install Library (jalankan sekali)

In [1]:
pip install neo4j

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import ast
import os
from neo4j import GraphDatabase

# ── Konfigurasi Koneksi ──────────────────────────────────
URI      = "neo4j://127.0.0.1:7687"
USERNAME = "neo4j"
PASSWORD = "password123"
DATABASE = "flimkg"

# ── Path Data ────────────────────────────────────────────
NODE_DIR = os.path.join('..', 'entity_construction', 'nodes')
REL_DIR  = os.path.join('..', 'entity_construction', 'relationships')

driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))
driver.verify_connectivity()
print('Koneksi ke Neo4j berhasil!')

Koneksi ke Neo4j berhasil!


## Helper: Batch Runner

In [3]:
def run_batch(query, data, batch_size=500):
    """Jalankan query Cypher dalam batch agar tidak overload memori."""
    total = len(data)
    for i in range(0, total, batch_size):
        batch = data[i:i+batch_size]
        with driver.session(database=DATABASE) as session:
            session.run(query, rows=batch)
        print(f'  Progress: {min(i+batch_size, total)}/{total}', end='\r')
    print(f'  Selesai: {total} rows diproses.')

---
## 1. Buat Constraints (Unique Index)
Memastikan tidak ada duplikasi node.

In [4]:
constraints = [
    "CREATE CONSTRAINT IF NOT EXISTS FOR (m:Movie)    REQUIRE m.movie_id    IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (a:Actor)    REQUIRE a.name        IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (d:Director) REQUIRE d.name        IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (g:Genre)    REQUIRE g.name        IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (k:Keyword)  REQUIRE k.name        IS UNIQUE",
]

with driver.session(database=DATABASE) as session:
    for c in constraints:
        session.run(c)

print('Constraints berhasil dibuat!')

Constraints berhasil dibuat!


---
## 2. Import Node: Movie

In [5]:
df_movie = pd.read_csv(os.path.join(NODE_DIR, 'nodes_movie.csv'))
df_movie['movie_id'] = df_movie['movie_id'].astype(int)
df_movie['revenue']  = pd.to_numeric(df_movie['revenue'], errors='coerce').fillna(0)
df_movie['release_date'] = df_movie['release_date'].fillna('').astype(str)

movie_data = df_movie.to_dict('records')

query_movie = """
UNWIND $rows AS row
MERGE (m:Movie {movie_id: row.movie_id})
SET m.title        = row.title,
    m.release_date = row.release_date,
    m.revenue      = row.revenue
"""

print(f'Import {len(movie_data)} Movie nodes...')
run_batch(query_movie, movie_data)

Import 45460 Movie nodes...
  Selesai: 45460 rows diproses.


---
## 3. Import Node: Actor

In [6]:
df_actor = pd.read_csv(os.path.join(NODE_DIR, 'nodes_actor.csv'))
actor_data = df_actor.to_dict('records')

query_actor = """
UNWIND $rows AS row
MERGE (a:Actor {name: row.name})
"""

print(f'Import {len(actor_data)} Actor nodes...')
run_batch(query_actor, actor_data)

Import 73155 Actor nodes...
  Selesai: 73155 rows diproses.


---
## 4. Import Node: Director

In [7]:
df_director = pd.read_csv(os.path.join(NODE_DIR, 'nodes_director.csv'))
director_data = df_director.to_dict('records')

query_director = """
UNWIND $rows AS row
MERGE (d:Director {name: row.name})
"""

print(f'Import {len(director_data)} Director nodes...')
run_batch(query_director, director_data)

Import 19740 Director nodes...
  Selesai: 19740 rows diproses.


---
## 5. Import Node: Genre

In [8]:
df_genre = pd.read_csv(os.path.join(NODE_DIR, 'nodes_genre.csv'))
genre_data = df_genre.to_dict('records')

query_genre = """
UNWIND $rows AS row
MERGE (g:Genre {name: row.name})
"""

print(f'Import {len(genre_data)} Genre nodes...')
run_batch(query_genre, genre_data)

Import 20 Genre nodes...
  Selesai: 20 rows diproses.


---
## 6. Import Node: Keyword

In [9]:
df_keyword = pd.read_csv(os.path.join(NODE_DIR, 'nodes_keyword.csv'))
keyword_data = df_keyword.to_dict('records')

query_keyword = """
UNWIND $rows AS row
MERGE (k:Keyword {name: row.name})
"""

print(f'Import {len(keyword_data)} Keyword nodes...')
run_batch(query_keyword, keyword_data)

Import 19956 Keyword nodes...
  Selesai: 19956 rows diproses.


---
## 7. Import Relationship: Movie → actedBy → Actor

In [10]:
df_rel_actor = pd.read_csv(os.path.join(REL_DIR, 'rels_movie_actor.csv'))
df_rel_actor['movie_id'] = df_rel_actor['movie_id'].astype(int)
rel_actor_data = df_rel_actor.to_dict('records')

query_rel_actor = """
UNWIND $rows AS row
MATCH (m:Movie   {movie_id:  row.movie_id})
MATCH (a:Actor   {name:      row.actor_name})
MERGE (m)-[:actedBy]->(a)
"""

print(f'Import {len(rel_actor_data)} relasi actedBy...')
run_batch(query_rel_actor, rel_actor_data)

Import 207535 relasi actedBy...
  Selesai: 207535 rows diproses.


---
## 8. Import Relationship: Movie → directedBy → Director

In [11]:
df_rel_dir = pd.read_csv(os.path.join(REL_DIR, 'rels_movie_director.csv'))
df_rel_dir['movie_id'] = df_rel_dir['movie_id'].astype(int)
rel_dir_data = df_rel_dir.to_dict('records')

query_rel_dir = """
UNWIND $rows AS row
MATCH (m:Movie    {movie_id:     row.movie_id})
MATCH (d:Director {name:         row.director_name})
MERGE (m)-[:directedBy]->(d)
"""

print(f'Import {len(rel_dir_data)} relasi directedBy...')
run_batch(query_rel_dir, rel_dir_data)

Import 50295 relasi directedBy...
  Selesai: 50295 rows diproses.


---
## 9. Import Relationship: Movie → hasGenre → Genre

In [12]:
df_rel_genre = pd.read_csv(os.path.join(REL_DIR, 'rels_movie_genre.csv'))
df_rel_genre['movie_id'] = df_rel_genre['movie_id'].astype(int)
rel_genre_data = df_rel_genre.to_dict('records')

query_rel_genre = """
UNWIND $rows AS row
MATCH (m:Movie {movie_id:  row.movie_id})
MATCH (g:Genre {name:      row.genre_name})
MERGE (m)-[:hasGenre]->(g)
"""

print(f'Import {len(rel_genre_data)} relasi hasGenre...')
run_batch(query_rel_genre, rel_genre_data)

Import 93342 relasi hasGenre...
  Selesai: 93342 rows diproses.


---
## 10. Import Relationship: Movie → hasKeyword → Keyword

In [13]:
df_rel_kw = pd.read_csv(os.path.join(REL_DIR, 'rels_movie_keyword.csv'))
df_rel_kw['movie_id'] = df_rel_kw['movie_id'].astype(int)
rel_kw_data = df_rel_kw.to_dict('records')

query_rel_kw = """
UNWIND $rows AS row
MATCH (m:Movie   {movie_id:   row.movie_id})
MATCH (k:Keyword {name:       row.keyword_name})
MERGE (m)-[:hasKeyword]->(k)
"""

print(f'Import {len(rel_kw_data)} relasi hasKeyword...')
run_batch(query_rel_kw, rel_kw_data)

Import 159437 relasi hasKeyword...
  Selesai: 159437 rows diproses.


---
## 11. Verifikasi — Cek Jumlah Node & Relationship

In [14]:
checks = {
    'Movie nodes':       'MATCH (m:Movie)    RETURN count(m) AS total',
    'Actor nodes':       'MATCH (a:Actor)    RETURN count(a) AS total',
    'Director nodes':    'MATCH (d:Director) RETURN count(d) AS total',
    'Genre nodes':       'MATCH (g:Genre)    RETURN count(g) AS total',
    'Keyword nodes':     'MATCH (k:Keyword)  RETURN count(k) AS total',
    'actedBy rels':      'MATCH ()-[r:actedBy]->()    RETURN count(r) AS total',
    'directedBy rels':   'MATCH ()-[r:directedBy]->() RETURN count(r) AS total',
    'hasGenre rels':     'MATCH ()-[r:hasGenre]->()   RETURN count(r) AS total',
    'hasKeyword rels':   'MATCH ()-[r:hasKeyword]->() RETURN count(r) AS total',
}

print('=== VERIFIKASI GRAPH ===')
with driver.session(database=DATABASE) as session:
    for label, query in checks.items():
        result = session.run(query).single()
        print(f'  {label:<22}: {result["total"]:>7}')

=== VERIFIKASI GRAPH ===
  Movie nodes           :   45430
  Actor nodes           :   73155
  Director nodes        :   19740
  Genre nodes           :      20
  Keyword nodes         :   19956
  actedBy rels          :  202103
  directedBy rels       :   48995
  hasGenre rels         :   91006
  hasKeyword rels       :  156582


---
## 12. Contoh Query Cypher — Cek Graph

In [15]:
# Contoh: lihat film beserta genre dan sutradara
query_sample = """
MATCH (m:Movie)-[:directedBy]->(d:Director),
      (m)-[:hasGenre]->(g:Genre)
RETURN m.title AS film, d.name AS sutradara, collect(g.name) AS genre
LIMIT 5
"""

with driver.session(database=DATABASE) as session:
    results = session.run(query_sample).data()

pd.DataFrame(results)

,film,sutradara,genre
0,Heart of Stone,Dale Trevillion\t,[Thriller]
1,One Man Force,Dale Trevillion\t,"[Action, Crime, Drama]"
2,The Legend of Kaspar Hauser,Davide Manuli,"[Comedy, Drama]"
3,Death Sentence,E.W. Swackhamer,"[Crime, Drama, Mystery, Thriller]"
4,Inseparable,Vitaliy Vorobyov,[Drama]


In [16]:
driver.close()
print('Koneksi ditutup.')

Koneksi ditutup.
